In [8]:
from pathlib import Path
import sys, json, joblib, pickle
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists(): PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "src").exists(), "Run from project root or notebooks/"
sys.path.insert(0,str(PROJECT_ROOT))
from src.features import add_features
DATA_DIR=PROJECT_ROOT/"data"
SUBMIT_DIR=PROJECT_ROOT/"submissions"
ARTIFACT_DIR=DATA_DIR/"artifacts"
ARTIFACT_DIR.mkdir(parents=True,exist_ok=True)
config=json.loads((ARTIFACT_DIR/"final_model_config.json").read_text(encoding="utf-8"))
FEATURE_COLS=config["feature_columns"]
CAT_COLS=config["categorical_columns"]
SELECTED=config["selected_model"]
print("Model:",SELECTED,"| feature set:",config["feature_set"])

Model: Weighted Blending | feature set: FS_selected


In [9]:
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
train_feat_path = DATA_DIR / "train_featured.pkl"
test_feat_path = DATA_DIR / "test_featured.pkl"

X_train_all = X_comp_test = None
if train_feat_path.exists() and test_feat_path.exists():
    try:
        X_train_all = pd.read_pickle(train_feat_path)
        X_comp_test = pd.read_pickle(test_feat_path)
        test_raw = pd.read_csv(DATA_DIR / "test.csv")
        print("Loaded compatible train/test feature caches.")
    except (NotImplementedError, ValueError, EOFError, pickle.UnpicklingError) as exc:
        X_train_all = X_comp_test = None
        print(f"Feature cache incompatible ({type(exc).__name__}); rebuilding from CSV.")

if X_train_all is None or X_comp_test is None:
    train_raw = pd.read_csv(DATA_DIR / "train.csv")
    test_raw = pd.read_csv(DATA_DIR / "test.csv")
    X_train_all = add_features(train_raw.drop(columns="is_fraud"))
    X_train_all["is_fraud"] = train_raw["is_fraud"].astype(int)
    X_train_all["id"] = train_raw["id"]
    X_comp_test = add_features(test_raw)
    X_comp_test["id"] = test_raw["id"]

y_all = X_train_all["is_fraud"].astype(int).to_numpy()

# Áp dụng Stratified 5-Fold trên toàn bộ tập train
from src.split import create_stratified_kfold
X_train_all = create_stratified_kfold(X_train_all, target_col="is_fraud", n_splits=5, shuffle=True, random_state=42, fold_col="fold")
folds = X_train_all["fold"].to_numpy()

X_train_all = X_train_all[FEATURE_COLS].copy()
X_comp_test = X_comp_test[FEATURE_COLS].copy()

for c in CAT_COLS:
    X_train_all[c] = X_train_all[c].astype("category")
    X_comp_test[c] = pd.Categorical(X_comp_test[c], categories=X_train_all[c].cat.categories)

print(f"Train shape: {X_train_all.shape} | Test shape: {X_comp_test.shape} | Folds: {np.unique(folds)}")


Feature cache incompatible (NotImplementedError); rebuilding from CSV.
Train shape: (699960, 31) | Test shape: (299984, 31) | Folds: [0 1 2 3 4]


In [10]:
USE_5FOLD_ENSEMBLE = True  # Huấn luyện 5 Folds và lấy trung bình dự đoán trên Test
test_preds = np.zeros(len(test_raw))
trained_models = []

if USE_5FOLD_ENSEMBLE:
    print(f"=== BẮT ĐẦU HUẤN LUYỆN STRATIFIED 5-FOLD ENSEMBLE ({SELECTED}) ===")
    for fold in range(5):
        tr_idx, val_idx = np.where(folds != fold)[0], np.where(folds == fold)[0]
        X_tr, y_tr = X_train_all.iloc[tr_idx], y_all[tr_idx]
        X_va, y_va = X_train_all.iloc[val_idx], y_all[val_idx]
        
        if SELECTED == "XGBoost":
            m = xgb.XGBClassifier(**config["xgboost_params"])
            m.fit(X_tr, y_tr)
            pred_va = m.predict_proba(X_va)[:, 1]
            pred_te = m.predict_proba(X_comp_test)[:, 1]
        elif SELECTED == "LightGBM":
            m = lgb.LGBMClassifier(**config["lightgbm_params"])
            m.fit(X_tr, y_tr)
            pred_va = m.predict_proba(X_va)[:, 1]
            pred_te = m.predict_proba(X_comp_test)[:, 1]
        elif SELECTED == "Weighted Blending":
            a = float(config["ensemble_weights"]["lightgbm"])
            m_lgb = lgb.LGBMClassifier(**config["lightgbm_params"]).fit(X_tr, y_tr)
            m_xgb = xgb.XGBClassifier(**config["xgboost_params"]).fit(X_tr, y_tr)
            pred_va = a * m_lgb.predict_proba(X_va)[:, 1] + (1 - a) * m_xgb.predict_proba(X_va)[:, 1]
            pred_te = a * m_lgb.predict_proba(X_comp_test)[:, 1] + (1 - a) * m_xgb.predict_proba(X_comp_test)[:, 1]
            m = {"lgb": m_lgb, "xgb": m_xgb}
            
        from sklearn.metrics import average_precision_score
        print(f"Fold {fold} Validation PR-AUC: {average_precision_score(y_va, pred_va):.6f}")
        test_preds += pred_te / 5.0
        trained_models.append(m)
    predictions = test_preds
else:
    # Huấn luyện 1 lần trên 100% dữ liệu
    if SELECTED == "XGBoost":
        m = xgb.XGBClassifier(**config["xgboost_params"]).fit(X_train_all, y_all)
        predictions = m.predict_proba(X_comp_test)[:, 1]
        trained_models = [m]
        
assert len(predictions) == len(test_raw) and np.isfinite(predictions).all() and ((predictions >= 0) & (predictions <= 1)).all()
print("\nThống kê phân phối xác suất dự đoán trên Test Set:")
print(pd.Series(predictions).describe(percentiles=[.01, .5, .95, .99]))


=== BẮT ĐẦU HUẤN LUYỆN STRATIFIED 5-FOLD ENSEMBLE (Weighted Blending) ===
Fold 0 Validation PR-AUC: 0.353881
Fold 1 Validation PR-AUC: 0.373773
Fold 2 Validation PR-AUC: 0.350534
Fold 3 Validation PR-AUC: 0.361815
Fold 4 Validation PR-AUC: 0.351003

Thống kê phân phối xác suất dự đoán trên Test Set:
count    299984.000000
mean          0.020143
std           0.064469
min           0.000002
1%            0.000047
50%           0.004223
95%           0.079997
99%           0.326043
max           0.995448
dtype: float64


In [11]:
target="is_fraud"
assert target in sample.columns, f"Expected {target} in sample_submission.csv, found {sample.columns.tolist()}"
if "id" in sample.columns:
    assert sample["id"].is_unique, "Duplicate ids in sample template"
    pred_lookup=pd.Series(predictions,index=test_raw["id"])
    assert set(sample["id"])==set(test_raw["id"]), "Submission IDs differ from competition test IDs"
    submission=sample.copy()
    submission[target]=submission["id"].map(pred_lookup)
else:
    assert len(sample)==len(test_raw), "Rows differ between template and competition test"
    submission=sample.copy()
    submission[target]=predictions
assert submission.columns.tolist()==sample.columns.tolist()
assert len(submission)==len(sample)
assert submission[target].notna().all()
assert submission[target].between(0,1).all()
assert np.isfinite(submission[target]).all()
output=SUBMIT_DIR/"submission.csv"
submission.to_csv(output,index=False)
print("Saved:",output,"shape:",submission.shape)
display(submission.head())

Saved: D:\Project\FraudDetection\submissions\submission.csv shape: (299984, 2)


,id,is_fraud
0,699960,0.000611
1,699961,0.000328
2,699962,0.250300
3,699963,0.004359
4,699964,0.004778


In [12]:
joblib.dump({"models":trained_models,"config":config,"features":FEATURE_COLS,
             "categoricals":CAT_COLS},ARTIFACT_DIR/"08_final_fitted_models.joblib")
(ARTIFACT_DIR/"08_submission_metadata.json").write_text(json.dumps({
    "selected_model":SELECTED,"train_rows":int(len(train_raw)),
    "test_rows":int(len(test_raw)),"feature_set":config["feature_set"],
    "submission_columns":submission.columns.tolist(),
    "prediction_min":float(predictions.min()),"prediction_max":float(predictions.max())
},indent=2),encoding="utf-8")
# Re-read to confirm the CSV really preserves schema
saved=pd.read_csv(output)
assert saved.shape==sample.shape and saved.columns.tolist()==sample.columns.tolist()
print("Verified submission:",output)

Verified submission: D:\Project\FraudDetection\submissions\submission.csv
